# Notebook 04 — Evaluation Analysis

This notebook analyzes the RAGAS evaluation results, compares baseline vs advanced pipelines,
performs category-based analysis, and visualizes the key findings.

**Prerequisites:** Run `python evaluation/run_evaluation.py` first to generate results.

In [ ]:
import sys
import json
from pathlib import Path
from collections import defaultdict

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Setup paths
sys.path.insert(0, str(Path('..').resolve()))

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

print("Setup complete!")

## 1. Load Evaluation Results

In [ ]:
# Load evaluation results
results_dir = Path('../evaluation/results')

with open(results_dir / 'eval_results.json', 'r') as f:
    results = json.load(f)

with open('../evaluation/eval_dataset.json', 'r') as f:
    dataset = json.load(f)

# Create mappings
id_to_info = {q['id']: q for q in dataset}

print(f"Modes evaluated: {list(results.keys())}")
print(f"Questions in dataset: {len(dataset)}")

## 2. Aggregate Metrics Comparison

In [ ]:
# Extract aggregate metrics
metrics_data = []
for mode, data in results.items():
    agg = data.get('aggregate', {})
    metrics_data.append({
        'Mode': mode.title(),
        'Faithfulness': agg.get('faithfulness', 0),
        'Answer Relevancy': agg.get('answer_relevancy', 0),
        'Context Recall': agg.get('context_recall', 0),
        'Context Precision': agg.get('context_precision', 0),
        'Avg Latency (s)': agg.get('avg_latency_seconds', 0)
    })

metrics_df = pd.DataFrame(metrics_data)
print("\n" + "="*60)
print("AGGREGATE METRICS COMPARISON")
print("="*60)
display(metrics_df.set_index('Mode').round(4))

In [ ]:
# Calculate improvements
if 'baseline' in results and 'advanced' in results:
    baseline = results['baseline']['aggregate']
    advanced = results['advanced']['aggregate']
    
    print("\n" + "="*60)
    print("IMPROVEMENT: Advanced vs Baseline")
    print("="*60)
    
    for metric in ['faithfulness', 'answer_relevancy', 'context_recall', 'context_precision']:
        base_val = baseline.get(metric, 0)
        adv_val = advanced.get(metric, 0)
        if base_val > 0:
            improvement = ((adv_val - base_val) / base_val) * 100
            print(f"{metric:20s}: {base_val:.4f} → {adv_val:.4f} ({improvement:+.1f}%)")
        else:
            print(f"{metric:20s}: {base_val:.4f} → {adv_val:.4f}")

## 3. Visualization: Metrics Comparison

In [ ]:
# Bar chart comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# RAGAS Metrics
metrics = ['Faithfulness', 'Answer Relevancy', 'Context Recall', 'Context Precision']
x = np.arange(len(metrics))
width = 0.35

if len(metrics_df) >= 2:
    baseline_vals = metrics_df[metrics_df['Mode'] == 'Baseline'][metrics].values[0] if 'Baseline' in metrics_df['Mode'].values else [0]*4
    advanced_vals = metrics_df[metrics_df['Mode'] == 'Advanced'][metrics].values[0] if 'Advanced' in metrics_df['Mode'].values else [0]*4
    
    bars1 = axes[0].bar(x - width/2, baseline_vals, width, label='Baseline', color='#3498DB')
    bars2 = axes[0].bar(x + width/2, advanced_vals, width, label='Advanced', color='#E74C3C')
    
    axes[0].set_ylabel('Score (0-1)')
    axes[0].set_title('RAGAS Metrics: Baseline vs Advanced')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(metrics, rotation=15, ha='right')
    axes[0].legend()
    axes[0].set_ylim(0, 1)
    
    # Add value labels
    for bar in bars1 + bars2:
        height = bar.get_height()
        axes[0].annotate(f'{height:.2f}',
                        xy=(bar.get_x() + bar.get_width() / 2, height),
                        xytext=(0, 3), textcoords="offset points",
                        ha='center', va='bottom', fontsize=9)
else:
    # Single mode visualization
    vals = metrics_df[metrics].values[0]
    axes[0].bar(x, vals, color='#E74C3C')
    axes[0].set_ylabel('Score (0-1)')
    axes[0].set_title('RAGAS Metrics (Advanced Mode)')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(metrics, rotation=15, ha='right')
    axes[0].set_ylim(0, 1)

# Latency comparison
latencies = metrics_df[['Mode', 'Avg Latency (s)']].set_index('Mode')
latencies.plot(kind='bar', ax=axes[1], color=['#3498DB', '#E74C3C'][:len(latencies)], legend=False)
axes[1].set_title('Average Response Latency')
axes[1].set_ylabel('Latency (seconds)')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('../evaluation/results/metrics_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nChart saved to: evaluation/results/metrics_comparison.png")

## 4. Category-Based Analysis

In [ ]:
# Category-based analysis (using advanced mode per-question data)
if 'per_question' in results.get('advanced', {}):
    advanced_data = results['advanced']['per_question']
    
    category_stats = defaultdict(lambda: {'count': 0, 'errors': 0, 'total_latency': 0})
    
    for item in advanced_data:
        cat = item.get('category', 'unknown')
        category_stats[cat]['count'] += 1
        category_stats[cat]['total_latency'] += item.get('latency_seconds', 0)
        if item.get('error') or 'error' in item.get('answer', '').lower():
            category_stats[cat]['errors'] += 1
    
    category_df = pd.DataFrame([
        {
            'Category': cat.replace('_', ' ').title(),
            'Questions': stats['count'],
            'Errors': stats['errors'],
            'Success Rate': f"{((stats['count'] - stats['errors']) / stats['count'] * 100):.0f}%",
            'Avg Latency (s)': round(stats['total_latency'] / stats['count'], 2)
        }
        for cat, stats in sorted(category_stats.items())
    ])
    
    print("\n" + "="*60)
    print("CATEGORY-BASED PERFORMANCE (Advanced Mode)")
    print("="*60)
    display(category_df.set_index('Category'))
else:
    print("No per-question data available for category analysis")

In [ ]:
# Category visualization
if 'category_df' in dir():
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Success rate by category
    success_rates = [(c['count'] - c['errors']) / c['count'] * 100 
                     for c in category_stats.values()]
    categories = [c.replace('_', ' ').title() for c in category_stats.keys()]
    
    colors = ['#2ECC71' if r >= 90 else '#F39C12' if r >= 70 else '#E74C3C' for r in success_rates]
    axes[0].barh(categories, success_rates, color=colors)
    axes[0].set_xlabel('Success Rate (%)')
    axes[0].set_title('Success Rate by Question Category')
    axes[0].set_xlim(0, 100)
    
    for i, v in enumerate(success_rates):
        axes[0].text(v + 2, i, f'{v:.0f}%', va='center')
    
    # Latency by category
    latencies = [c['total_latency'] / c['count'] for c in category_stats.values()]
    axes[1].barh(categories, latencies, color='#9B59B6')
    axes[1].set_xlabel('Average Latency (seconds)')
    axes[1].set_title('Average Latency by Question Category')
    
    for i, v in enumerate(latencies):
        axes[1].text(v + 0.5, i, f'{v:.1f}s', va='center')
    
    plt.tight_layout()
    plt.savefig('../evaluation/results/category_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("\nChart saved to: evaluation/results/category_analysis.png")

## 5. Error Analysis

In [ ]:
# Identify and categorize errors
if 'per_question' in results.get('advanced', {}):
    advanced_data = results['advanced']['per_question']
    
    error_cases = []
    for item in advanced_data:
        qid = item['id']
        answer = item.get('answer', '')
        ground_truth = id_to_info[qid]['ground_truth']
        
        error_type = None
        
        if item.get('error') or 'error' in answer.lower()[:50]:
            error_type = 'GENERATION_FAILURE'
        elif 'insufficient' in answer.lower() or 'do not contain' in answer.lower():
            error_type = 'RETRIEVAL_FAILURE'
        elif qid == 'q007' and ('62,020' in answer or '62020' in answer):
            error_type = 'RETRIEVAL_FAILURE'
        elif 'derived from' in answer.lower() and 'sum' in answer.lower():
            error_type = 'RETRIEVAL_INEFFICIENCY'
        
        if error_type:
            error_cases.append({
                'ID': qid,
                'Category': item.get('category', '').replace('_', ' ').title(),
                'Error Type': error_type,
                'Question': item['question'][:50] + '...',
                'Contexts': len(item.get('contexts', []))
            })
    
    if error_cases:
        error_df = pd.DataFrame(error_cases)
        print("\n" + "="*60)
        print(f"ERROR ANALYSIS ({len(error_cases)} issues found)")
        print("="*60)
        display(error_df)
        
        # Error type distribution
        print("\nError Type Distribution:")
        print(error_df['Error Type'].value_counts())
    else:
        print("No errors detected!")

In [ ]:
# Error distribution pie chart
if 'error_df' in dir() and len(error_df) > 0:
    fig, ax = plt.subplots(figsize=(8, 6))
    
    error_counts = error_df['Error Type'].value_counts()
    colors = {'RETRIEVAL_FAILURE': '#E74C3C', 'GENERATION_FAILURE': '#F39C12', 'RETRIEVAL_INEFFICIENCY': '#3498DB'}
    pie_colors = [colors.get(e, '#95A5A6') for e in error_counts.index]
    
    wedges, texts, autotexts = ax.pie(error_counts.values, labels=error_counts.index, 
                                       autopct='%1.0f%%', colors=pie_colors,
                                       explode=[0.05]*len(error_counts))
    ax.set_title(f'Error Type Distribution ({len(error_df)} total errors)')
    
    plt.tight_layout()
    plt.savefig('../evaluation/results/error_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("\nChart saved to: evaluation/results/error_distribution.png")

## 6. Detailed Per-Question Results

In [ ]:
# Create detailed results table
if 'per_question' in results.get('advanced', {}):
    detailed_rows = []
    for item in results['advanced']['per_question']:
        qid = item['id']
        info = id_to_info[qid]
        
        answer = item.get('answer', '')
        status = 'OK'
        if item.get('error') or 'error' in answer.lower()[:50]:
            status = 'ERROR'
        elif 'insufficient' in answer.lower():
            status = 'INSUFFICIENT'
        
        detailed_rows.append({
            'ID': qid,
            'Category': info['category'].replace('_', ' ').title()[:15],
            'Question': item['question'][:40] + '...',
            'Status': status,
            'Latency': f"{item.get('latency_seconds', 0):.1f}s",
            'Contexts': len(item.get('contexts', []))
        })
    
    detailed_df = pd.DataFrame(detailed_rows)
    print("\n" + "="*60)
    print("DETAILED PER-QUESTION RESULTS")
    print("="*60)
    display(detailed_df)

## 7. Latency Analysis

In [ ]:
# Latency distribution
if 'per_question' in results.get('advanced', {}):
    latencies = [item.get('latency_seconds', 0) for item in results['advanced']['per_question']]
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Histogram
    axes[0].hist(latencies, bins=15, color='#9B59B6', edgecolor='white')
    axes[0].axvline(np.mean(latencies), color='red', linestyle='--', label=f'Mean: {np.mean(latencies):.1f}s')
    axes[0].axvline(np.median(latencies), color='green', linestyle='--', label=f'Median: {np.median(latencies):.1f}s')
    axes[0].set_xlabel('Latency (seconds)')
    axes[0].set_ylabel('Number of queries')
    axes[0].set_title('Response Latency Distribution')
    axes[0].legend()
    
    # Box plot by category
    category_latencies = defaultdict(list)
    for item in results['advanced']['per_question']:
        cat = item.get('category', 'unknown').replace('_', ' ').title()
        category_latencies[cat].append(item.get('latency_seconds', 0))
    
    bp = axes[1].boxplot(category_latencies.values(), labels=category_latencies.keys(), patch_artist=True)
    for patch in bp['boxes']:
        patch.set_facecolor('#9B59B6')
        patch.set_alpha(0.7)
    axes[1].set_ylabel('Latency (seconds)')
    axes[1].set_title('Latency Distribution by Category')
    axes[1].tick_params(axis='x', rotation=20)
    
    plt.tight_layout()
    plt.savefig('../evaluation/results/latency_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"\nLatency Statistics:")
    print(f"  Min:    {min(latencies):.2f}s")
    print(f"  Max:    {max(latencies):.2f}s")
    print(f"  Mean:   {np.mean(latencies):.2f}s")
    print(f"  Median: {np.median(latencies):.2f}s")
    print(f"  Std:    {np.std(latencies):.2f}s")
    
    print("\nChart saved to: evaluation/results/latency_analysis.png")

## 8. Executive Summary

In [ ]:
print("\n" + "="*70)
print("                    EVALUATION EXECUTIVE SUMMARY")
print("="*70)

if 'baseline' in results and 'advanced' in results:
    baseline = results['baseline']['aggregate']
    advanced = results['advanced']['aggregate']
    
    print("\n KEY FINDINGS:")
    print("-" * 50)
    
    faith_imp = ((advanced['faithfulness'] - baseline['faithfulness']) / baseline['faithfulness']) * 100
    print(f"  1. Faithfulness improved {faith_imp:.1f}% ({baseline['faithfulness']:.3f} → {advanced['faithfulness']:.3f})")
    
    rel_imp = ((advanced['answer_relevancy'] - baseline['answer_relevancy']) / baseline['answer_relevancy']) * 100
    print(f"  2. Answer Relevancy improved {rel_imp:.1f}% ({baseline['answer_relevancy']:.3f} → {advanced['answer_relevancy']:.3f})")
    
    recall_imp = ((advanced['context_recall'] - baseline['context_recall']) / baseline['context_recall']) * 100
    print(f"  3. Context Recall improved {recall_imp:.1f}% ({baseline['context_recall']:.3f} → {advanced['context_recall']:.3f})")
    
    lat_change = advanced['avg_latency_seconds'] - baseline['avg_latency_seconds']
    print(f"  4. Latency increased by {lat_change:.1f}s ({baseline['avg_latency_seconds']:.1f}s → {advanced['avg_latency_seconds']:.1f}s)")
    
    print("\n RECOMMENDATIONS:")
    print("-" * 50)
    print("  ✓ Use Advanced (Hybrid+Rerank) for production deployment")
    print("  ✓ Accept ~2s latency trade-off for significant quality gains")
    print("  ✓ Focus improvement efforts on temporal query handling")
    print("  ✓ Consider larger context window for multi-period queries")
else:
    # Single mode summary
    mode = list(results.keys())[0]
    agg = results[mode]['aggregate']
    print(f"\n Results for {mode.title()} mode:")
    for k, v in agg.items():
        print(f"  {k}: {v:.4f}")

print("\n" + "="*70)